# Data Preprocessing

## Objective

The objective of this notebook is to prepare the dataset for exploratory analysis and machine learning by addressing the data quality issues identified in the previous stage.

The preprocessing pipeline includes handling invalid values, treating missing values, addressing anomalous observations, managing extreme outliers, and validating the cleaned dataset before model development.

The resulting cleaned dataset will be used for Business EDA, Feature Engineering, and Model Training.

In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)

In [4]:
credit_df = pd.read_csv("../data/raw/GiveMeSomeCredit-training.csv")

In [5]:
credit_clean = credit_df.copy()

# Preprocessing Plan

Based on the Data Quality Assessment, the following issues were identified:

| Feature | Issue | Planned Treatment |
|---------|-------|-------------------|
| Age | One invalid value (0) | Remove the invalid record |
| Delinquency Features | Values 96 and 98 | Replace with appropriate values after investigation |
| MonthlyIncome | 19.82% missing | Median imputation |
| NumberOfDependents | 2.62% missing | Median imputation |
| RevolvingUtilizationOfUnsecuredLines | Extreme outliers | Investigate transformation or capping |
| DebtRatio | Extreme outliers | Investigate transformation or capping |

Each preprocessing step will be validated before proceeding to the next stage.


## Problem

During the Data Quality Assessment, one record was found with an age value of **0**, which is not a valid age for a loan applicant.

## Treatment

Since this represents only **one record (0.00067% of the dataset)**, it is removed from the dataset rather than being imputed.

This ensures that only valid customer records are retained for subsequent analysis.

In [6]:
# Number of rows before removing invalid age
print(f"Rows before removal: {credit_clean.shape[0]}")

# Number of invalid age records
invalid_age = (credit_clean["age"] == 0).sum()
print(f"Invalid age records: {invalid_age}")

# Remove invalid age records
credit_clean = credit_clean[credit_clean["age"] != 0].copy()

# Verify
print(f"Rows after removal: {credit_clean.shape[0]}")

# Final check
print(f"Remaining invalid age records: {(credit_clean['age'] == 0).sum()}")

Rows before removal: 150000
Invalid age records: 1
Rows after removal: 149999
Remaining invalid age records: 0


### Result

- Successfully removed the single record with an invalid age value.
- The dataset now contains **149,999** valid customer records.
- No remaining records have an age value of **0**.

# 2. Handle Anomalous Delinquency Values

## Problem

During the Data Quality Assessment, the delinquency-related features were found to contain anomalous values of **96** and **98**. These values are unlikely to represent the actual number of late payment occurrences and appear to be placeholder or coded values.

## Treatment

The anomalous values are replaced with **missing values (NaN)** rather than being removed or arbitrarily modified.

This preserves all customer records while allowing the missing values to be handled consistently during the imputation stage.

In [7]:
# Delinquency columns
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

# Count anomalous values before replacement
for col in delinquency_cols:
    count = credit_clean[col].isin([96, 98]).sum()
    print(f"{col}: {count} anomalous values")

# Replace 96 and 98 with NaN
credit_clean[delinquency_cols] = credit_clean[delinquency_cols].replace([96, 98], np.nan)

# Verify replacement
print("\nAfter replacement:\n")

for col in delinquency_cols:
    print(f"{col}")
    print(credit_clean[col].isna().sum())
    print("-" * 50)

NumberOfTime30-59DaysPastDueNotWorse: 269 anomalous values
NumberOfTime60-89DaysPastDueNotWorse: 269 anomalous values
NumberOfTimes90DaysLate: 269 anomalous values

After replacement:

NumberOfTime30-59DaysPastDueNotWorse
269
--------------------------------------------------
NumberOfTime60-89DaysPastDueNotWorse
269
--------------------------------------------------
NumberOfTimes90DaysLate
269
--------------------------------------------------


# Step 3: Missing Value Treatment

## Objective

The dataset contains missing values in several features. Since the nature and distribution of these features differ, a single imputation strategy is not appropriate for all variables.

The following strategies are adopted:

| Feature | Imputation Strategy | Justification |
|---------|---------------------|---------------|
| MonthlyIncome | Median | Highly right-skewed with extreme outliers |
| NumberOfDependents | Median | Numerical count feature with low missing percentage |
| Delinquency Features | Mode | Count-based variables where zero is the most frequent value |

In [8]:
missing_before = credit_clean.isnull().sum()

missing_before[missing_before > 0].sort_values(ascending=False)

MonthlyIncome                           29731
NumberOfDependents                       3924
NumberOfTime30-59DaysPastDueNotWorse      269
NumberOfTimes90DaysLate                   269
NumberOfTime60-89DaysPastDueNotWorse      269
dtype: int64

In [9]:
# Monthly Income
credit_clean["MonthlyIncome"] = credit_clean["MonthlyIncome"].fillna(
    credit_clean["MonthlyIncome"].median()
)

# Number of Dependents
credit_clean["NumberOfDependents"] = credit_clean["NumberOfDependents"].fillna(
    credit_clean["NumberOfDependents"].median()
)

In [12]:
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for col in delinquency_cols:
    credit_clean[col] = credit_clean[col].fillna(
        credit_clean[col].mode()[0]
    )

In [13]:
missing_after = credit_clean.isnull().sum()

missing_after[missing_after > 0]

Series([], dtype: int64)

In [15]:
check_point_1= credit_clean
check_point_1

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2.0,0.802982,9120.0,13,0.0,6,0.0,2.0
1,2,0,0.957151,40,0.0,0.121876,2600.0,4,0.0,0,0.0,1.0
2,3,0,0.658180,38,1.0,0.085113,3042.0,2,1.0,0,0.0,0.0
3,4,0,0.233810,30,0.0,0.036050,3300.0,5,0.0,0,0.0,0.0
4,5,0,0.907239,49,1.0,0.024926,63588.0,7,0.0,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
149995,149996,0,0.040674,74,0.0,0.225131,2100.0,4,0.0,1,0.0,0.0
149996,149997,0,0.299745,44,0.0,0.716562,5584.0,4,0.0,1,0.0,2.0
149997,149998,0,0.246044,58,0.0,3870.000000,5400.0,18,0.0,1,0.0,0.0
149998,149999,0,0.000000,30,0.0,0.000000,5716.0,4,0.0,0,0.0,0.0
